In [1]:
# ── Cell 1: Install & Imports ─────────────────────────────────────────────────

!pip install -q transformers datasets scikit-learn huggingface_hub

from datasets import load_dataset
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from huggingface_hub import login
import torch
import torch.nn as nn
import numpy as np
import gc
import shutil

print("All imports done.")

All imports done.


In [2]:
# ── Cell 2: Data Loading + Label Coarsening ───────────────────────────────────

dataset  = load_dataset("freococo/burmese-contextual-pragmatics")
train_ds = dataset["train"]

# ── Politeness: 13 → 6 ───────────────────────────────────────────────────────
politeness_map = {
    "neutral":"neutral", "polite":"polite", "informal":"informal",
    "professional":"professional", "blunt":"blunt", "rude":"rude",
    "very_polite":"polite", "friendly":"informal", "religious":"polite",
    "impolite":"rude", "stern":"blunt", "sarcastic":"rude",
    "polite_but_firm":"polite"
}

# ── Tone: 131 → 5 (Appraisal Theory) ────────────────────────────────────────
positive = {"Warm","Sweet","Cheerful","Soft","Sincere","Friendly","Kind",
            "Caring","Gentle","Encouraging","Supportive","Peaceful","Grateful",
            "Romantic","Joyful","Hopeful","Welcoming","Trusting","Approving",
            "Complimentary","Amused","Excited","Moved","Nostalgic","Sympathetic",
            "Pitying","Curious","Playful","Tender","Awe-struck","Awe",
            "Grateful/Humble","Care"}
formal   = {"Professional","Serious","Respectful","Humble","Stern","Firm",
            "Determined","Authoritative","Commanding","Objective","Deep",
            "Intense","Stern/Concerned"}
negative = {"Angry","Annoyed","Hostile","Aggressive","Harsh","Hateful",
            "Disgusted","Cold","Dismissive","Sarcastic","Impolite","Rude",
            "Blunt","Crude","Envious","Dangerous","Fed up","Dry","Indifferent"}
emotional= {"Urgent","Pleading","Dramatic","Sad","Distressed","Tired",
            "Exhausted","Panicked","Shocked","Surprised","Concerned",
            "Apologetic","Appologetic","Regretful","Emotional","Teasing",
            "Casual","Dazed","Confused","Alarmed","Overwhelmed","Startled",
            "Devastated","Worried","Nervous/Sincere","Sad/Soft","Sad/Sweet",
            "Distress","Painful","Sorrowful","Weak","Uncertain","Weary",
            "Hurry","Impatient","Relieved","Fearful","Scared","Frightened",
            "Grieved","Incredulous","Bored","Lazy","Sleepy","Stressed",
            "Exasperated","Frustrated","Mischievous","Suggestive","Spooky",
            "Whispering","Indirect","Exclamatory","Lazy/Neutral",
            "Casual/Sleepy","Cheerful/Tired","Sweet/Weary","Sad/Serious",
            "Soft/Vulnerable","Natural","Considerate","Faithful","Trusting"}

def coarsen_tone(t):
    if t in positive:  return "positive"
    if t in formal:    return "formal"
    if t in negative:  return "negative"
    if t in emotional: return "emotional"
    return "neutral"

# ── Power relation: 8 → 4 ────────────────────────────────────────────────────
power_map = {
    "equal":"equal",
    "inferior_to_superior":"inferior_to_superior",
    "superior_to_inferior":"superior_to_inferior",
    "any":"any",
    "customer_to_staff":"inferior_to_superior",
    "customer_to_seller":"inferior_to_superior",
    "passenger_to_driver":"inferior_to_superior",
    "patient_to_doctor":"inferior_to_superior",
}

# ── Apply all mappings ────────────────────────────────────────────────────────
def apply_all_labels(example):
    example["politeness_coarse"] = politeness_map.get(example["politeness"], "neutral")
    example["tone_coarse"]       = coarsen_tone(example["tone"])
    example["power_coarse"]      = power_map.get(example["power_relation"], "equal")
    example["register_label"]    = example["register"]
    return example

train_ds = train_ds.map(apply_all_labels)

# ── Fit label encoders ────────────────────────────────────────────────────────
le_pol  = LabelEncoder().fit(train_ds["politeness_coarse"])
le_reg  = LabelEncoder().fit(train_ds["register_label"])
le_pow  = LabelEncoder().fit(train_ds["power_coarse"])
le_tone = LabelEncoder().fit(train_ds["tone_coarse"])

print("Politeness classes:", list(le_pol.classes_))
print("Register classes:  ", list(le_reg.classes_))
print("Power classes:     ", list(le_pow.classes_))
print("Tone classes:      ", list(le_tone.classes_))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Politeness classes: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]
Register classes:   [np.str_('colloquial'), np.str_('formal'), np.str_('slang'), np.str_('standard')]
Power classes:      [np.str_('any'), np.str_('equal'), np.str_('inferior_to_superior'), np.str_('superior_to_inferior')]
Tone classes:       [np.str_('emotional'), np.str_('formal'), np.str_('negative'), np.str_('neutral'), np.str_('positive')]


In [3]:
# ── Cell 3: Encode Labels + Splits + Class Weights ────────────────────────────

def encode_all_labels(example):
    example["label_pol"]  = int(le_pol.transform([example["politeness_coarse"]])[0])
    example["label_reg"]  = int(le_reg.transform([example["register_label"]])[0])
    example["label_pow"]  = int(le_pow.transform([example["power_coarse"]])[0])
    example["label_tone"] = int(le_tone.transform([example["tone_coarse"]])[0])
    return example

train_ds = train_ds.map(encode_all_labels)

# ── Same seed=42 split as all your previous notebooks ────────────────────────
split1     = train_ds.train_test_split(test_size=0.30, seed=42)
split2     = split1["test"].train_test_split(test_size=0.50, seed=42)
train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

# ── Class weights ─────────────────────────────────────────────────────────────
def get_weights(data, label_col, n_classes):
    counts = Counter(data[label_col])
    total  = len(data)
    return torch.tensor(
        [total / (n_classes * counts[i]) for i in range(n_classes)],
        dtype=torch.float
    )

w_pol  = get_weights(train_data, "label_pol",  len(le_pol.classes_))
w_reg  = get_weights(train_data, "label_reg",  len(le_reg.classes_))
w_pow  = get_weights(train_data, "label_pow",  len(le_pow.classes_))
w_tone = get_weights(train_data, "label_tone", len(le_tone.classes_))

print("Weights pol: ", [round(x,3) for x in w_pol.tolist()])
print("Weights reg: ", [round(x,3) for x in w_reg.tolist()])
print("Weights pow: ", [round(x,3) for x in w_pow.tolist()])
print("Weights tone:", [round(x,3) for x in w_tone.tolist()])

Train: 1540  Val: 330  Test: 330
Weights pol:  [5.238, 0.976, 0.318, 0.817, 3.889, 6.111]
Weights reg:  [0.364, 4.01, 3.565, 1.39]
Weights pow:  [3.377, 0.33, 2.567, 3.5]
Weights tone: [1.149, 1.257, 4.738, 0.487, 0.936]


In [4]:
# ── Cell 4: Tokenizer + Tokenization Functions ────────────────────────────────

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

KEEP_COLS = ["input_ids", "attention_mask", "label"]

def tokenize(data, fn, label_col):
    def wrapped(example):
        enc = fn(example)
        enc["label"] = example[label_col]
        return enc
    cols_to_remove = [c for c in data.column_names if c not in KEEP_COLS]
    out = data.map(wrapped, remove_columns=cols_to_remove)
    out.set_format("torch")
    return out

# ── Classifier A — Register (utterance only, unchanged) ──────────────────────
def tok_reg(example):
    return tokenizer(
        example["utterance"],
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

# ── Classifier B_v2 — Power (utterance + context + predicted register) ────────
# At TRAINING time: uses ground truth register (teacher forcing)
# At INFERENCE time: we'll swap in predicted register from Classifier A
def tok_pow(example):
    text = (f"[register: {example['register_label']}] "
            + example["utterance"]
            + " </s> " + str(example["context"]).strip())
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

# ── Classifier C_v2 — Tone (utterance + context + predicted register + power) ─
# At TRAINING time: uses ground truth register + power (teacher forcing)
# At INFERENCE time: we'll swap in predictions from A and B
def tok_tone(example):
    text = (f"[register: {example['register_label']}] "
            f"[power: {example['power_coarse']}] "
            + example["utterance"]
            + " </s> " + str(example["context"]).strip())
    return tokenizer(
        text,
        max_length=MAX_LENGTH, truncation=True, padding="max_length"
    )

print("Tokenizer ready:", type(tokenizer).__name__)
print("tok_reg  → Classifier A (utterance only)")
print("tok_pow  → Classifier B_v2 (+ ground truth register at train time)")
print("tok_tone → Classifier C_v2 (+ ground truth register + power at train time)")

Tokenizer ready: XLMRobertaTokenizer
tok_reg  → Classifier A (utterance only)
tok_pow  → Classifier B_v2 (+ ground truth register at train time)
tok_tone → Classifier C_v2 (+ ground truth register + power at train time)


In [5]:
# ── Cell 5: Trainer Setup + HF Login ─────────────────────────────────────────

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        loss    = nn.CrossEntropyLoss(
                      weight=self.class_weights.to(outputs.logits.device)
                  )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy":    round(accuracy_score(labels, preds), 4),
        "macro_f1":    round(f1_score(labels, preds, average="macro"), 4),
        "weighted_f1": round(f1_score(labels, preds, average="weighted"), 4),
    }

def make_training_args(hub_repo):
    return TrainingArguments(
        output_dir=hub_repo,
        num_train_epochs=8,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        warmup_steps=150,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=50,
        report_to="none",
        seed=42,
        push_to_hub=True,
        hub_model_id=hub_repo,
        hub_strategy="every_save"
    )

login()

print("Setup complete.")

Setup complete.


In [6]:
# ── Colab B Cell 6: Classifier C_v2 — Tone (with register + power prefix) ────

print("Tokenizing for Classifier C_v2...")
tok_train_C = tokenize(train_data, tok_tone, "label_tone")
tok_val_C   = tokenize(val_data,   tok_tone, "label_tone")
tok_test_C  = tokenize(test_data,  tok_tone, "label_tone")
print("Done.")

model_C = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(le_tone.classes_)
)

trainer_C = WeightedTrainer(
    class_weights=w_tone,
    model=model_C,
    args=make_training_args("annasus10/xlmr-burmese-tone-v2"),
    train_dataset=tok_train_C,
    eval_dataset=tok_val_C,
    compute_metrics=compute_metrics
)

trainer_C.train()
trainer_C.push_to_hub()
print("Classifier C_v2 pushed.")

Tokenizing for Classifier C_v2...


Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Done.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tone-v2/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...tone-v2/model.safetensors:  14%|#3        |  152MB / 1.11GB            

Classifier C_v2 pushed.


In [7]:
# ── Eval Classifier C_v2 on test set ─────────────────────────────────────────
from sklearn.metrics import classification_report
import numpy as np

results_C = trainer_C.evaluate(tok_test_C)
print("\nClassifier C_v2 (Tone) Test Results:")
for k, v in results_C.items():
    print(f"  {k}: {v}")

preds_C_out = trainer_C.predict(tok_test_C)
preds_C = np.argmax(preds_C_out.predictions, axis=1)
true_C  = preds_C_out.label_ids

print("\nClassifier C_v2 Classification Report:")
print(classification_report(true_C, preds_C, target_names=le_tone.classes_))


Classifier C_v2 (Tone) Test Results:
  eval_loss: 1.1462364196777344
  eval_accuracy: 0.6545
  eval_macro_f1: 0.6212
  eval_weighted_f1: 0.6581
  eval_runtime: 2.3673
  eval_samples_per_second: 139.399
  eval_steps_per_second: 8.871
  epoch: 8.0

Classifier C_v2 Classification Report:
              precision    recall  f1-score   support

   emotional       0.55      0.67      0.61        55
      formal       0.70      0.66      0.68        58
    negative       0.54      0.44      0.48        16
     neutral       0.82      0.62      0.70       133
    positive       0.54      0.76      0.63        68

    accuracy                           0.65       330
   macro avg       0.63      0.63      0.62       330
weighted avg       0.68      0.65      0.66       330

